<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>05. 📝 Markdown to HTML Converter</b>
</div>


### 📌 Project Overview
In this project, we write a customized Markdown to HTML parser from scratch.
Using regular expressions and structural state tracks, this parser translates standard heading symbols, inline bold/italic wrappers, links, code snippets, and unordered lists into fully styled, browser-ready HTML documents.

#### 🔑 Key Concepts Covered:
- Utilizing Python's `re` module for lexical string replacements
- Reading unstructured text files and processing patterns line-by-line
- Implementing text parsing state machines to handle nested HTML block states (like lists and block quotes)
- Generating and saving files programmatically


In [1]:
import re

class MarkdownParser:
    def __init__(self):
        # Compile inline formatting patterns
        self.inline_rules = [
            (r'\*\*(.*?)\*\*', r'<strong>\1</strong>'),  # Bold (**text**)
            (r'\*(.*?)\*', r'<em>\1</em>'),              # Italic (*text*)
            (r'`(.*?)`', r'<code>\1</code>'),            # Inline Code (`code`)
            (r'\[(.*?)\]\((.*?)\)', r'<a href="\2">\1</a>') # Links ([text](url))
        ]
        
    def parse_inline(self, text):
        for pattern, replacement in self.inline_rules:
            text = re.sub(pattern, replacement, text)
        return text
        
    def convert(self, markdown_text):
        lines = markdown_text.split('\n')
        html_output = []
        in_list = False
        in_code_block = False
        code_lines = []
        
        for line in lines:
            # Check block-level code markers
            if line.strip().startswith('```'):
                if in_code_block:
                    in_code_block = False
                    html_output.append('<pre><code>' + '\n'.join(code_lines) + '</code></pre>')
                    code_lines = []
                else:
                    in_code_block = True
                continue
                
            if in_code_block:
                code_lines.append(line)
                continue
                
            # Check unordered list items
            if line.strip().startswith('- '):
                if not in_list:
                    html_output.append('<ul>')
                    in_list = True
                item_content = line.strip()[2:]
                html_output.append(f'  <li>{self.parse_inline(item_content)}</li>')
                continue
            elif in_list:
                html_output.append('</ul>')
                in_list = False
                
            # Skip blank lines
            if not line.strip():
                continue
                
            # Check markdown heading symbols (# to ######)
            header_match = re.match(r'^(#{1,6})\s+(.*)$', line)
            if header_match:
                level = len(header_match.group(1))
                content = self.parse_inline(header_match.group(2))
                html_output.append(f'<h{level}>{content}</h{level}>')
                continue
                
            # Process standard paragraph lines
            html_output.append(f'<p>{self.parse_inline(line.strip())}</p>')
            
        if in_list:
            html_output.append('</ul>')
        return '\n'.join(html_output)


In [2]:
markdown_content = '''# My Markdown Parsing Test

This is a **pure-Python markdown parser** converting markdown syntax to *HTML* documents.
It supports:
- **Formatted** words
- Inline `code` tags
- Hyperlinks to [GitHub](https://github.com)

Here is a code block:
```python
def test_parser():
    print("Success!")
```
'''

parser = MarkdownParser()
html_result = parser.convert(markdown_content)
print('--- PARSED HTML OUTPUT ---')
print(html_result)

output_filename = 'markdown_demo.html'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(f'<html><body style="font-family:sans-serif;padding:30px;line-height:1.6;">{html_result}</body></html>')
print(f'\n✅ Saved generated page to "{output_filename}". Open it in a browser to inspect!')


--- PARSED HTML OUTPUT ---
<h1>My Markdown Parsing Test</h1>
<p>This is a <strong>pure-Python markdown parser</strong> converting markdown syntax to <em>HTML</em> documents.</p>
<p>It supports:</p>
<ul>
  <li><strong>Formatted</strong> words</li>
  <li>Inline <code>code</code> tags</li>
  <li>Hyperlinks to <a href="https://github.com">GitHub</a></li>
</ul>
<p>Here is a code block:</p>
<pre><code>def test_parser():
    print("Success!")</code></pre>

✅ Saved generated page to "markdown_demo.html". Open it in a browser to inspect!
